# Histopathology Image Classification Using Deep Learning | Python | TensorFlow | 2026

## Project Overview

This project develops a deep learning workflow for binary classification of histopathology images from the **BreakHis** dataset into **benign** and **malignant** categories.

**Technologies:** Python, TensorFlow, Keras, NumPy, Pandas, Scikit-learn, Matplotlib, Pillow  
**Model:** Convolutional Neural Network (CNN)  
**Dataset:** BreakHis — Fold 1  
**Image size:** 128 × 128 pixels

> **Methodology note:** BreakHis Fold 1 is used as the held-out test split. Duplicate filename entries are removed before creating the train/validation data so that the same image is not placed in both training and testing sets.

## 1. Import Libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay
)

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import ModelCheckpoint

print("TensorFlow version:", tf.__version__)
print("Available GPUs:", tf.config.list_physical_devices("GPU"))

## 2. Load the BreakHis Dataset

In [ ]:
dataset_path = "/kaggle/input/datasets/ambarish/breakhis/BreaKHis_v1/BreaKHis_v1/histology_slides/breast"
folds_path = "/kaggle/input/datasets/ambarish/breakhis/Folds.csv"

image_paths = []

for root, dirs, files in os.walk(dataset_path):
    for file in files:
        if file.lower().endswith(".png"):
            image_paths.append(os.path.join(root, file))

print("Total image files found:", len(image_paths))

folds_df = pd.read_csv(folds_path)
print("Folds shape:", folds_df.shape)
print("Columns:", list(folds_df.columns))

## 3. Build Image-Path Lookup and Select Fold 1

In [ ]:
path_lookup = {
    os.path.basename(path): path
    for path in image_paths
}

folds_df["image_path"] = folds_df["filename"].map(path_lookup)

if folds_df["image_path"].isna().any():
    raise ValueError("Some CSV filenames could not be matched to image files.")

# Fold 1 train/test groups
fold1_train = folds_df[
    (folds_df["fold"] == 1) &
    (folds_df["grp"] == "train")
].drop_duplicates("filename").copy()

fold1_test = folds_df[
    (folds_df["fold"] == 1) &
    (folds_df["grp"] == "test")
].drop_duplicates("filename").copy()

# Binary labels: 0 = benign, 1 = malignant
fold1_train["label"] = fold1_train["filename"].apply(
    lambda x: 0 if "/benign/" in x else 1
)
fold1_test["label"] = fold1_test["filename"].apply(
    lambda x: 0 if "/benign/" in x else 1
)

print("Fold 1 training images:", len(fold1_train))
print("Fold 1 test images:", len(fold1_test))
print("Total unique images:", len(fold1_train) + len(fold1_test))
print("\nTraining labels:")
print(fold1_train["label"].value_counts().rename(index={0: "Benign", 1: "Malignant"}))
print("\nTest labels:")
print(fold1_test["label"].value_counts().rename(index={0: "Benign", 1: "Malignant"}))

## 4. Create Training and Validation Sets

In [ ]:
proper_train, proper_val = train_test_split(
    fold1_train,
    test_size=0.20,
    random_state=42,
    stratify=fold1_train["label"]
)

print("Training images:", len(proper_train))
print("Validation images:", len(proper_val))
print("Test images:", len(fold1_test))

## 5. Image Preprocessing

In [ ]:
IMG_SIZE = 128
BATCH_SIZE = 32

def create_dataset(dataframe, shuffle=False):
    dataset = tf.data.Dataset.from_tensor_slices(
        (
            dataframe["image_path"].values,
            dataframe["label"].values
        )
    )

    def load_image(path, label):
        image = tf.io.read_file(path)
        image = tf.image.decode_png(image, channels=3)
        image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])
        image = image / 255.0
        return image, label

    dataset = dataset.map(
        load_image,
        num_parallel_calls=tf.data.AUTOTUNE
    )

    if shuffle:
        dataset = dataset.shuffle(buffer_size=1000, seed=42)

    dataset = dataset.batch(BATCH_SIZE)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)

    return dataset

proper_train_dataset = create_dataset(proper_train, shuffle=True)
proper_val_dataset = create_dataset(proper_val)
proper_test_dataset = create_dataset(fold1_test)

print("Datasets created successfully.")

## 6. Build the Convolutional Neural Network

In [ ]:
proper_cnn_model = models.Sequential([
    layers.Input(shape=(128, 128, 3)),

    layers.Conv2D(32, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(128, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),

    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.5),

    layers.Dense(1, activation="sigmoid")
])

proper_cnn_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

proper_cnn_model.summary()

## 7. Train the CNN

In [ ]:
checkpoint = ModelCheckpoint(
    "/kaggle/working/breakhis_cnn.keras",
    monitor="val_accuracy",
    save_best_only=True,
    mode="max",
    verbose=1
)

history_proper = proper_cnn_model.fit(
    proper_train_dataset,
    epochs=5,
    validation_data=proper_val_dataset,
    callbacks=[checkpoint]
)

## 8. Training and Validation Accuracy

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    history_proper.history["accuracy"],
    label="Training Accuracy"
)
plt.plot(
    history_proper.history["val_accuracy"],
    label="Validation Accuracy"
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training and Validation Accuracy")
plt.legend()
plt.tight_layout()

plt.savefig(
    "/kaggle/working/training_validation_accuracy.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

## 9. Training and Validation Loss

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    history_proper.history["loss"],
    label="Training Loss"
)
plt.plot(
    history_proper.history["val_loss"],
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.tight_layout()

plt.savefig(
    "/kaggle/working/training_validation_loss.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

## 10. Evaluate the Final Model on the Untouched Fold 1 Test Set

In [ ]:
# Load the best validation model saved during training
proper_cnn_model = tf.keras.models.load_model(
    "/kaggle/working/breakhis_cnn.keras"
)

test_loss, test_accuracy = proper_cnn_model.evaluate(
    proper_test_dataset,
    verbose=1
)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

## 11. Accuracy, Precision, Recall and F1-Score

In [ ]:
y_true = []
y_pred_prob = []

for images, labels_batch in proper_test_dataset:
    predictions = proper_cnn_model.predict(images, verbose=0).ravel()
    y_pred_prob.extend(predictions)
    y_true.extend(labels_batch.numpy())

y_true = np.array(y_true)
y_pred_prob = np.array(y_pred_prob)
y_pred = (y_pred_prob >= 0.5).astype(int)

accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-score : {f1:.4f}")

## 12. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_true, y_pred)

print("Confusion Matrix:")
print(cm)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Benign", "Malignant"]
)

disp.plot()
plt.title("CNN Confusion Matrix - BreakHis Fold 1")
plt.tight_layout()

plt.savefig(
    "/kaggle/working/confusion_matrix.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

## 13. Example Single-Image Prediction

In [ ]:
sample_path = fold1_test["image_path"].iloc[0]
true_label = fold1_test["label"].iloc[0]

img = Image.open(sample_path).convert("RGB")
img_resized = img.resize((IMG_SIZE, IMG_SIZE))

img_array = np.array(img_resized) / 255.0
img_array = np.expand_dims(img_array, axis=0)

prediction = proper_cnn_model.predict(
    img_array,
    verbose=0
)[0][0]

predicted_label = "Malignant" if prediction >= 0.5 else "Benign"
actual_label = "Malignant" if true_label == 1 else "Benign"

plt.figure(figsize=(6, 6))
plt.imshow(img)
plt.axis("off")
plt.title(
    f"Actual: {actual_label}\n"
    f"Predicted: {predicted_label}\n"
    f"Malignant Probability: {prediction:.2%}"
)
plt.show()

## 14. Final Results

The final saved CNN model was evaluated on the untouched **BreakHis Fold 1 test set (2,904 images)**.

| Metric | Result |
|---|---:|
| Test Accuracy | **82.71%** |
| Precision | **82.27%** |
| Recall | **93.72%** |
| F1-score | **87.62%** |
| Test Loss | **0.4619** |

**Confusion matrix:**

- True Negative (Benign): 625
- False Positive: 383
- False Negative: 119
- True Positive (Malignant): 1,777

> This project is for computational/research and educational purposes. The model performs image classification on the BreakHis dataset and is not a clinical diagnostic system.